# A tour of connexplorer

One package for the FlyWire FAFB (v783) and Male CNS (v1.0) connectomes. Everything returns polars DataFrames or plain numpy/scipy objects; root ids are the public identifiers; nothing opens a browser.

Sections:

1. Open a dataset and select neurons
2. Connectivity, with the T4/T5 motion-detector cells as the running example
3. Graph statistics and a second dataset
4. Physical synapse locations
5. Morphology and a cable model of CT1, with its synapses drawn on the skeleton
6. Viewer links and the command line

Build the datasets first (`01_build_datasets.ipynb` or `connexplorer build ...`) and put the FlyWire skeleton zip under `data/flywire_783/tables/skeletons/`.

In [ ]:
import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.express as px

import connexplorer as cnx

ds = cnx.open("flywire")   # by name (searches ./data and $CONNEXPLORER_DATA) or by path; reads only the manifest
ds.info()

## 1. Tables and selection

`ds.cells` and `ds.types` are cached polars frames. The eight T4/T5 types are the ON (T4) and OFF (T5) direction-selective cells of the optic lobe, one subtype per cardinal direction.

In [ ]:
t45_types = ds.types.filter(pl.col("type").str.contains(r"^T[45][a-d]$"))
t45_types.select("type", "n_cells", "nt", "nt_source", "family", "subsystem")

In [ ]:
t4 = ds[["T4a", "T4b", "T4c", "T4d"]]        # NeuronSet: union of types
t5 = ds[["T5a", "T5b", "T5c", "T5d"]]
t4a_R = ds.select(type="T4a", side="R")        # column == value filters, or any polars expression
n = t4a_R[0]                                    # a Neuron is a NeuronSet of size one
print(t4, t5, t4a_R, n, sep="\n")
print("root ids:", n.root_id, t4a_R.ids[:3], "| set algebra:", len(t4 | t5), len(t4 & t4a_R))
n.cell

## 2. Connectivity: who talks to T4 and T5?

Partner tables come straight from the edge table held as a CSR matrix. `by="type"` groups partners; `normalize=True` adds `frac_input` (share of the set's own input), `frac_partner_output` (share of that partner type's whole-dataset output landing on the set) and their geometric mean `weight_norm`.

In [ ]:
print("T4 input types")
display(t4.inputs(by="type", normalize=True).head(8))
print("T5 input types")
display(t5.inputs(by="type", normalize=True).head(8))
print("T4 output types")
t4.outputs(by="type").head(8)

### Type-to-type matrix

The precomputed type matrix is indexed by type names. Here: the strongest input types of T4 and T5 (rows) onto the eight subtypes (columns), as a share of each subtype's input.

In [ ]:
post = t45_types["type"].to_list()
pre = (
    pl.concat([t4.inputs(by="type").head(8), t5.inputs(by="type").head(8)])
    .drop_nulls("type").group_by("type").agg(pl.col("n_syn").sum()).sort("n_syn", descending=True)["type"].to_list()
)
blk = ds.connectivity.types[pre, post]
print(blk)
frac = blk.values / ds.connectivity.type_totals.in_syn[[ds.types["type"].to_list().index(t) for t in post]]
fig = px.imshow(100 * frac, x=post, y=pre, color_continuous_scale="Viridis", labels=dict(color="% of input"), aspect="auto",
                title="Input to T4/T5 subtypes by presynaptic type (% of the subtype's total input)")
fig.show()
blk.long.head()

### Cell-level blocks and one neuron

`ds.connectivity[pre, post]` takes type names, NeuronSets or root ids and gives a block with dense, sparse, long and wide views. Types are contiguous id ranges on disk, so a type block is a slice, not a gather.

In [ ]:
blk = ds.connectivity["T4a", "LPi14"]      # 1457 T4a cells onto the 4 LPi14 cells
print(blk, blk.values.shape, blk.sparse.nnz, "nonzero pairs")
display(blk.long.head())
display(n.outputs(min_syn=5).head())          # threshold on the per-partner total, never a neuropil fragment
display(n.inputs(by="neuropil"))
n.partners().head()

In [ ]:
# Self-connections are kept in the data and masked by one flag (default off)
print("autapses off:", n.outputs().height, "partners;", ds.connectivity.types["T4a", "T4a"], "T4a->T4a synapses")
ds.connectivity.autapses = True
print("autapses on: ", n.outputs().height, "partners;", ds.connectivity.types["T4a", "T4a"], "T4a->T4a synapses")
ds.connectivity.autapses = False

## 3. Graph statistics and a second dataset

Degrees and synapse totals per cell (whole dataset, or restricted to the set with `within=True`), hubs, reciprocal pairs, degree distributions.

In [ ]:
display(t4.hubs(k=5, by="out_syn"))
rec = ds["T4a"].reciprocal()
print(rec.height, "reciprocal T4a-T4a pairs; strongest:")
display(rec.head(3))
dist = t4.degree_distribution("in")
px.bar(dist.to_pandas(), x="degree", y="n_cells", title="In-degree of T4 cells (partners anywhere in the brain)").show()
t4.summary()

In [ ]:
mc = cnx.open("mcns")
print(mc, "| Male CNS types mapping to FlyWire R7:", mc.types_like("R7"))
cnx.compare(ds["T4a"], mc["T4a"]).inputs().head(8)     # aligned on FlyWire type names

## 4. Physical synapse locations

The synapse table stays on disk (80 M rows). Queries on a cell or a pair of sets read only the row groups that can contain them, so a neuron's synapses come back in a few milliseconds.

In [ ]:
syn_in = n.synapses("in")                     # pre, post (root ids), x_nm, y_nm, z_nm, neuropil
syn_out = n.synapses("out")
print(syn_in.height, "inputs,", syn_out.height, "outputs of", n)
display(syn_in.head())
mi1_t4a = ds.synapses.between("Mi1", "T4a")
print(mi1_t4a.height, "Mi1 -> T4a synapses;", mi1_t4a["neuropil"].value_counts().sort("count", descending=True).head(3).rows())
xyz = cnx.xyz(mi1_t4a.sample(4000, seed=0)) / 1e3   # micrometres
fig = go.Figure(go.Scatter3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="markers", marker=dict(size=1.5, color=xyz[:, 2], colorscale="Viridis")))
fig.update_layout(title="4000 of the Mi1 -> T4a synapses (um)", scene=dict(aspectmode="data"), height=500)
fig.show()

## 5. Morphology: CT1 and where its synapses sit

CT1 is one giant GABAergic cell per hemisphere that contacts every T4/T5 column. Its skeleton has 400k nodes and 100 mm of cable; it loads in under a second from the SWC zip and is always in micrometres.

In [ ]:
import navis

ct1 = ds.select(type="CT1", side="R")[0]
sk = ct1.skeleton()
print(ct1, "|", sk.n_nodes, "nodes,", f"{float(sk.cable_length) / 1e3:.1f} mm of cable,", sk.units)
display(ct1.inputs(by="type").head(5))
ct1.outputs(by="type").head(5)

In [ ]:
# Skeleton (downsampled for drawing) with 3000 input and 3000 output synapses, in micrometres
sk_small = navis.downsample_neuron(sk, 25, inplace=False)
fig = navis.plot3d(sk_small, backend="plotly", inline=False, color="#888888")
for df, name, color in ((ct1.synapses("in"), "inputs (post)", "#d62728"), (ct1.synapses("out"), "outputs (pre)", "#1f77b4")):
    p = cnx.xyz(df.sample(3000, seed=0)) / 1e3
    fig.add_trace(go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], mode="markers", marker=dict(size=1.5, color=color), name=name))
fig.update_layout(title="CT1 (right) with a sample of its synapses", scene=dict(aspectmode="data"), height=650)
fig.show()

### Compartments and a passive cable model

`segment` makes one compartment per navis segment; compartments shorter than `min_length_um` are merged into their parent, so the conductance matrix is never singular. The cable model factorizes its matrix once, so repeated injections and time steps are cheap.

In [ ]:
comp = cnx.morph.segment(sk, min_length_um=2.0)
comp.summary()
m = cnx.models.Cable(comp, Rm=8000, Ra=400, Cm=0.6)
distal = int(comp.table["depth"].arg_max())          # the deepest compartment
V = m.steady_state({distal: 10e-12})                 # 10 pA, millivolts per compartment
print(f"injection at compartment {distal}: {V[distal]:.2f} mV there, {V[0]:.4f} mV at the root, input resistance {m.input_resistance(distal):.2f} GOhm")
print(m.attenuation(distal))
c = comp.centers
fig = go.Figure(go.Scatter3d(x=c[:, 0], y=c[:, 1], z=c[:, 2], mode="markers", marker=dict(size=2, color=np.log10(V), colorscale="Plasma", colorbar=dict(title="log10 mV"))))
fig.update_layout(title=f"Steady-state voltage across CT1's {len(comp)} compartments (log scale)", scene=dict(aspectmode="data"), height=600)
fig.show()

In [ ]:
# A 20 ms, 10 pA pulse at the same site: implicit Euler on the factorized matrix
pulse = np.zeros(100); pulse[10:30] = 10e-12
Vt = m.transient({distal: pulse}, dt=1e-3)
px.line(x=np.arange(100), y=Vt[distal], labels=dict(x="ms", y="mV"), title="Voltage at the injection site").show()

## 6. Viewer links, on-demand Male CNS skeletons, command line

`view` returns a Neuroglancer URL (spelunker by default; also `cave`, `flywire`, `codex`). `partners` adds the top partner types as coloured layers and `synapses=True` their synapse points.

The Male CNS skeletons are too large to keep as a set; `mc[body].skeleton()` fetches that one body from the vendor's public store the first time and keeps it as an SWC file under `data/mcns_v1.0/tables/skeletons/`.

In [ ]:
url = n.view(partners="in", top=3, synapses=True)
print(url[:110], "...")
print(ds["T4a"][:3].view(viewer="codex"))
mi1 = mc["Mi1"][0]
sk_mc = mi1.skeleton()                              # fetched on first use (a few kB)
print(mi1, "|", sk_mc.n_nodes, "nodes,", f"{float(sk_mc.cable_length):.0f} um; the store has no radii, so every node gets {sk_mc.nodes.radius.iloc[0]} um")

The same queries are available from the shell:

```
connexplorer datasets
connexplorer query flywire T4a -t 5
connexplorer query flywire 720575940599755718 -m inputs --by cell
connexplorer neuron mcns 12473
connexplorer view flywire CT1 --partners in --top 3 --synapses
```